# Official Simulink PV model: FMI event-lifecycle compatibility

The original MathWorks **PVArrayGridAverageModelExample** retains its PV array, MPPT, average boost converter, DC link, PLL and current-control loops. Only current-output taps were added for measurement. It is fully discrete: the FMI 3.0 Model Exchange export has **zero continuous states**, no input ports and a next internal event every 50 microseconds.

This notebook validates the standalone `FMI_DiscreteProbe` event replay against the original Simulink simulation and an independent FMPy 0.3.32 replay. **This is not a DPsim MNA or trapezoidal integration benchmark.** The coupled larger-converter goal remains unfinished.

All files are under the SSD development directory. The six-second original simulation has 120001 saved samples. No trace shifting, parameter fitting or source-model simplification is applied.

In [ ]:
from pathlib import Path
import subprocess, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path("/Volumes/SSDMatthias/development/dpsim")
ROOT = REPO / "outputs/fmi-nonlinear/pv"
FMU = Path(
    "/Volumes/SSDMatthias/development/fmi-external-benchmarks/matlab/PVArrayGridAverageFMUProbe/PVArrayGridAverageModel.fmu"
)
EXE = REPO / "build-macos-check/dpsim-fmi/FMI_DiscreteProbe"
folder = ROOT / "discrete-probe"
folder.mkdir(exist_ok=True)
env = os.environ.copy()
env["TMPDIR"] = str(REPO / "outputs/fmi-nonlinear/tmp")
with (folder / "run.log").open("w") as log:
    subprocess.run(
        [str(EXE), str(FMU), "6", ".00005"],
        cwd=folder,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
        check=True,
    )


def read(path):
    d = pd.read_csv(path)
    return d.rename(
        columns={"time_s": "time", **{"i_" + p + "_A": "i_" + p for p in "abc"}}
    )


native = read(ROOT / "reference_simulink.csv")
independent = read(ROOT / "reference_fmpy.csv")
imported = read(folder / "discrete_probe.csv")
for d in [native, independent, imported]:
    assert len(d) == 120001 and np.isfinite(d.to_numpy()).all()
    assert np.allclose(d.time, imported.time, rtol=0, atol=1e-12)
rows = []
for label, reference in [
    ("Original Simulink", native),
    ("Independent FMPy", independent),
]:
    for p in "abc":
        error = imported["i_" + p] - np.interp(
            imported.time, reference.time, reference["i_" + p]
        )
        row = dict(
            reference=label,
            phase=p,
            max_error_A=float(np.max(abs(error))),
            rms_error_A=float(np.sqrt(np.mean(error**2))),
        )
        assert row["max_error_A"] < 1e-8
        rows.append(row)
metrics = pd.DataFrame(rows)
metrics.to_csv(ROOT / "comparison_metrics.csv", index=False)
display(metrics)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9))
for label, d, style in [
    ("Original Simulink", native, "-"),
    ("Independent FMPy", independent, ":"),
    ("Standalone FMI replay", imported, "--"),
]:
    magnitude = np.sqrt(sum(d["i_" + p] ** 2 for p in "abc") / 3)
    axes[0].plot(d.time, magnitude, style, label=label, lw=1.2)
    axes[1].plot(d.time, d.i_a, style, label=label, lw=1.2)
axes[0].set(xlabel="Time [s]", ylabel="abc current magnitude [A]", xlim=(0, 6))
local = native.loc[native.time.between(1.99, 2.04), "i_a"]
limit = 1.15 * local.abs().max()
axes[1].set(
    xlabel="Time [s]",
    ylabel="Phase-A current [A]",
    xlim=(1.99, 2.04),
    ylim=(-limit, limit),
)
axes[0].legend()
for p in "abc":
    axes[2].plot(
        imported.time,
        1e12 * (imported["i_" + p] - native["i_" + p]),
        label="Phase " + p,
    )
axes[2].set(xlabel="Time [s]", ylabel="Replay minus Simulink [pA]")
axes[2].legend()
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(ROOT / "compatibility.png", dpi=150, bbox_inches="tight")
plt.show()

## Coupled-import limitation

A provisional `GetFloat64` before the sampled event changes this export's subsequent history. An isolated FMPy experiment reproduces it; restoring the advertised FMI state also leaves a discrepancy. Consequently, agreement of the accepted-step replay does **not** establish that this FMU is safe for Newton probes.

A continuous-powergui copy, retaining all original sampled controls, fails MATLAB Model Exchange export because its `sfun_spid_contc` network S-function is non-inlined. Replacing it with an ordinary state-space block is overwritten during model update. The physical network also contains a switch, so freezing one `power_analyze` matrix is not a valid general replacement. No such simplification is used in these results.

Reproduction scripts and diagnostic logs are in `outputs/fmi-nonlinear`: `export_pv_probe.m`, `simulate_pv_example.m`, `pv/fmpy_reference.py`, `pv/protocol_variants.py`, `pv-continuous-export.log` and `pv-replace-sps.log`. FMPy and lxml are installed in a separate `fmi-external-benchmarks/python-reference` directory on the SSD. The original official example remains unchanged.

See `dpsim-fmi/NONLINEAR.md` for the implemented importer subset and the successful coupled generator and Modelica magnetic-circuit benchmarks.